
# Memory-Optimized Temporally Correlated Multi-Image Richardson–Lucy

This notebook implements a **memory-optimized multi-image Richardson–Lucy (RL) deconvolution** for repeated fluorescence images.

The new regularization uses the **temporal-correlation principle only**:

- the same underlying fluorescence structure is present in all repeated frames,
- frame-to-frame noise is assumed to be approximately independent,
- therefore a true RL correction tends to be reproducible across frames,
- while noise-driven corrections have approximately zero cross-frame correlation.

There are **no radiality maps, gradients, local spatial correlations, or SRRF-style spatial feature extraction**.

---

## Core idea

For each frame \(l\), calculate the standard RL correction

\[
C_l^{(k)}
=
\frac{
H^T
\left[
\dfrac{y_l}{Hx^{(k)}+b_l+\epsilon}
\right]
}{
H^T\mathbf{1}+\epsilon
}.
\]

Define the correction deviation from the neutral RL update:

\[
d_l^{(k)} = C_l^{(k)} - 1.
\]

If

\[
d_l=s+n_l
\]

where \(s\) is a reproducible correction and \(n_l\) is independent zero-mean temporal noise, then for two different frames

\[
E[d_l d_m] = s^2, \qquad l\neq m,
\]

because

\[
E[n_l n_m]\approx 0.
\]

The mean off-diagonal temporal product is

\[
Q
=
\frac{2}{L(L-1)}
\sum_{l<m} d_l d_m.
\]

Rather than storing all \(L\) correction maps, this notebook computes \(Q\) using streaming Welford statistics:

\[
Q
=
\bar d^2 - \frac{s_d^2}{L},
\]

where \(s_d^2\) is the sample temporal variance of the correction deviations.

A correlation-derived coherent correction amplitude is then

\[
d_{\mathrm{corr}}
=
\mathrm{sign}(\bar d)\sqrt{\max(Q,0)}.
\]

The final update blends ordinary multi-image RL with the temporal-correlation estimate:

\[
d_{\mathrm{update}}
=
(1-\lambda)\bar d
+
\lambda d_{\mathrm{corr}},
\]

\[
\boxed{
x^{(k+1)}
=
x^{(k)}
\left(1+d_{\mathrm{update}}\right)
}
\]

with \(0\leq\lambda\leq1\).

- \(\lambda=0\): standard multi-image RL
- \(\lambda=1\): fully temporal-correlation-based update
- intermediate values: conservative blend

This is an **experimental regularized RL method**. The temporal-correlation modification is not the standard Poisson maximum-likelihood RL update and should be validated against conventional RL controls.


## 1. Install packages in JupyterLite

In [ ]:

# JupyterLite / Pyodide package installation.
# Run this cell once when opening the notebook in JupyterLite.

import piplite
await piplite.install(["numpy", "matplotlib", "scipy", "tifffile"])


## 2. Imports

In [ ]:

import gc
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tifffile as tiff
from scipy.signal import fftconvolve

EPS = np.float32(1e-7)



## 3. Memory-efficient TIFF reader

The reader below does **not load the entire TIFF stack into RAM** when the TIFF contains multiple pages. Each frame is read only when required.

This is useful for JupyterLite, where the Python kernel runs inside the browser and memory is limited.


In [ ]:

class TiffStackReader:
    def __init__(self, path):
        self.path = str(path)
        self.tf = tiff.TiffFile(self.path)
        self.pages = self.tf.pages
        self._cached_stack = None

        if len(self.pages) > 1:
            first = self.pages[0].asarray()
            if first.ndim != 2:
                raise ValueError("Expected 2-D grayscale TIFF pages.")
            self.shape = first.shape
            self.n_frames = len(self.pages)
            self.mode = "pages"

        else:
            # Fallback for TIFF files stored as one 3-D series.
            arr = self.tf.asarray()
            if arr.ndim == 2:
                arr = arr[None, ...]
            if arr.ndim != 3:
                raise ValueError(
                    f"Expected a 2-D image or 3-D stack, got shape {arr.shape}."
                )
            self._cached_stack = arr
            self.n_frames = arr.shape[0]
            self.shape = arr.shape[1:]
            self.mode = "cached_3d"

        print(
            f"Opened {self.path}\n"
            f"  frames: {self.n_frames}\n"
            f"  image shape: {self.shape}\n"
            f"  reader mode: {self.mode}"
        )

    def get_frame(self, i):
        if i < 0 or i >= self.n_frames:
            raise IndexError(i)

        if self.mode == "pages":
            frame = self.pages[i].asarray()
        else:
            frame = self._cached_stack[i]

        frame = np.asarray(frame, dtype=np.float32)

        if frame.shape != self.shape:
            raise ValueError(f"Frame {i} has unexpected shape {frame.shape}.")

        frame[~np.isfinite(frame)] = 0
        np.maximum(frame, 0, out=frame)
        return frame

    def close(self):
        self.tf.close()

    def __len__(self):
        return self.n_frames


## 4. PSF and convolution helpers

In [ ]:

def normalize_psf(psf):
    psf = np.asarray(psf, dtype=np.float32)
    psf = np.clip(psf, 0, None)
    s = float(psf.sum())
    if s <= 0:
        raise ValueError("PSF sum must be > 0.")
    psf /= s
    return psf


def gaussian_psf(size=21, sigma=2.0):
    if size % 2 == 0:
        size += 1

    ax = np.arange(-(size // 2), size // 2 + 1, dtype=np.float32)
    xx, yy = np.meshgrid(ax, ax)

    psf = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    return normalize_psf(psf)


def conv2(image, kernel):
    # Convert output back to float32 immediately to reduce retained memory.
    return fftconvolve(image, kernel, mode="same").astype(np.float32, copy=False)


def adjoint_conv2(image, psf):
    return fftconvolve(
        image,
        psf[::-1, ::-1],
        mode="same"
    ).astype(np.float32, copy=False)


def make_sensitivity(image_shape, psf):
    ones = np.ones(image_shape, dtype=np.float32)
    sensitivity = adjoint_conv2(ones, psf)
    del ones
    return np.maximum(sensitivity, EPS)


## 5. Streaming stack statistics and background estimation

In [ ]:

def streaming_mean(reader):
    mean_img = np.zeros(reader.shape, dtype=np.float32)

    for i in range(reader.n_frames):
        frame = reader.get_frame(i)
        mean_img += (frame - mean_img) / np.float32(i + 1)
        del frame

    return mean_img


def estimate_scalar_backgrounds(reader, percentile=5.0):
    backgrounds = np.empty(reader.n_frames, dtype=np.float32)

    for i in range(reader.n_frames):
        frame = reader.get_frame(i)
        backgrounds[i] = np.percentile(frame, percentile)
        del frame

    return backgrounds


def prepare_backgrounds(backgrounds, reader):
    if backgrounds is None:
        return np.zeros(reader.n_frames, dtype=np.float32)

    b = np.asarray(backgrounds, dtype=np.float32)

    if b.ndim == 0:
        return np.full(reader.n_frames, float(b), dtype=np.float32)

    if b.ndim == 1 and len(b) == reader.n_frames:
        return b

    raise ValueError(
        "For the memory-optimized implementation, backgrounds must be "
        "None, one scalar, or one scalar per frame."
    )



## 6. Standard streaming multi-image RL

This implementation is a control. It processes one frame at a time and accumulates only the mean RL correction.

It therefore avoids creating an array with shape `(frames, y, x)`.


In [ ]:

def streaming_multi_image_rl(
    reader,
    psf,
    iterations=20,
    backgrounds=None,
    init=None,
    eps=EPS,
    verbose=True,
):
    psf = normalize_psf(psf)
    b = prepare_backgrounds(backgrounds, reader)

    sensitivity = make_sensitivity(reader.shape, psf)

    if init is None:
        x = streaming_mean(reader)
        x -= np.float32(np.mean(b))
        np.maximum(x, eps, out=x)
    else:
        x = np.asarray(init, dtype=np.float32).copy()
        np.maximum(x, eps, out=x)

    L = reader.n_frames

    for k in range(iterations):
        mean_correction = np.zeros(reader.shape, dtype=np.float32)

        for l in range(L):
            y = reader.get_frame(l)

            pred = conv2(x, psf)
            pred += b[l]
            np.maximum(pred, eps, out=pred)

            ratio = y / pred
            correction = adjoint_conv2(ratio, psf)
            correction /= sensitivity

            # Streaming temporal mean.
            mean_correction += (
                correction - mean_correction
            ) / np.float32(l + 1)

            del y, pred, ratio, correction

        x *= mean_correction
        np.maximum(x, 0, out=x)

        if verbose:
            print(
                f"Iteration {k+1:3d}/{iterations}: "
                f"mean(x)={float(x.mean()):.4g}"
            )

        del mean_correction
        gc.collect()

    return x



## 7. Temporally correlated multi-image RL

This is the new algorithm.

At each pixel, Welford's streaming algorithm calculates:

- temporal mean of \(d_l=C_l-1\),
- temporal variance of \(d_l\),

without retaining the correction maps from previous frames.

The off-diagonal pairwise temporal product

\[
Q =
\frac{2}{L(L-1)}\sum_{l<m}d_l d_m
\]

is obtained from

\[
Q = \bar d^2-\frac{s_d^2}{L}.
\]

If the correction contains a temporally reproducible component \(s\) and independent noise, \(Q\) estimates \(s^2\).


In [ ]:

def temporally_correlated_multi_image_rl(
    reader,
    psf,
    iterations=20,
    backgrounds=None,
    correlation_strength=0.75,
    init=None,
    eps=EPS,
    verbose=True,
    return_history=False,
):
    """
    Memory-optimized temporally correlated multi-image RL.

    Parameters
    ----------
    reader : TiffStackReader
        Streaming TIFF reader.

    psf : 2-D array
        Common PSF for all repeated acquisitions.

    iterations : int
        Number of RL iterations.

    backgrounds : None, scalar, or 1-D array
        Background for each frame.

    correlation_strength : float in [0, 1]
        lambda = 0 -> standard multi-image RL.
        lambda = 1 -> fully correlation-derived correction amplitude.

    init : 2-D array or None
        Initial latent image.

    return_history : bool
        Return diagnostics when True.

    Notes
    -----
    No spatial feature extraction is added.
    The only new regularization is computed across the temporal/frame axis.
    """

    lam = float(correlation_strength)
    if not (0.0 <= lam <= 1.0):
        raise ValueError("correlation_strength must be between 0 and 1.")

    if reader.n_frames < 2:
        raise ValueError("At least two frames are required.")

    psf = normalize_psf(psf)
    b = prepare_backgrounds(backgrounds, reader)

    sensitivity = make_sensitivity(reader.shape, psf)

    if init is None:
        x = streaming_mean(reader)
        x -= np.float32(np.mean(b))
        np.maximum(x, eps, out=x)
    else:
        x = np.asarray(init, dtype=np.float32).copy()
        np.maximum(x, eps, out=x)

    L = reader.n_frames

    history = {
        "relative_change": [],
        "mean_temporal_coherence": [],
        "positive_cross_fraction": [],
    }

    for k in range(iterations):

        # Welford accumulators.
        # Only two image-sized temporal-statistics arrays are retained.
        mean_d = np.zeros(reader.shape, dtype=np.float32)
        M2_d = np.zeros(reader.shape, dtype=np.float32)

        for l in range(L):
            y = reader.get_frame(l)

            pred = conv2(x, psf)
            pred += b[l]
            np.maximum(pred, eps, out=pred)

            ratio = y / pred

            correction = adjoint_conv2(ratio, psf)
            correction /= sensitivity

            # d = correction - neutral RL correction (1)
            correction -= np.float32(1.0)

            # Pixelwise Welford update along time only.
            n = np.float32(l + 1)

            delta = correction - mean_d
            mean_d += delta / n
            delta2 = correction - mean_d
            M2_d += delta * delta2

            del y, pred, ratio, correction, delta, delta2

        # Sample temporal variance.
        sample_var = M2_d / np.float32(L - 1)

        # Mean off-diagonal temporal cross-product:
        # Q = mean_{l != m}(d_l d_m)
        cross = mean_d * mean_d - sample_var / np.float32(L)
        np.maximum(cross, 0, out=cross)

        # Correlation-derived coherent correction amplitude.
        coherent_d = np.sqrt(cross, dtype=np.float32)
        coherent_d *= np.sign(mean_d)

        # Blend conventional multi-image RL correction with
        # the temporal-correlation-derived correction.
        if lam == 0.0:
            update_d = mean_d
        elif lam == 1.0:
            update_d = coherent_d
        else:
            update_d = (
                np.float32(1.0 - lam) * mean_d
                + np.float32(lam) * coherent_d
            )

        update = np.float32(1.0) + update_d
        np.maximum(update, eps, out=update)

        if return_history:
            x_old = x.copy()

        x *= update
        np.maximum(x, 0, out=x)

        if return_history:
            rel = np.linalg.norm(x - x_old) / max(
                float(np.linalg.norm(x_old)),
                float(eps),
            )

            # A diagnostic temporal coherence score:
            # cross / total temporal second moment.
            population_var = M2_d / np.float32(L)
            total_energy = mean_d * mean_d + population_var
            coherence = cross / np.maximum(total_energy, eps)

            history["relative_change"].append(float(rel))
            history["mean_temporal_coherence"].append(float(np.mean(coherence)))
            history["positive_cross_fraction"].append(
                float(np.mean(cross > 0))
            )

            del x_old, population_var, total_energy, coherence

        if verbose:
            print(
                f"Iteration {k+1:3d}/{iterations}: "
                f"mean(x)={float(x.mean()):.4g}, "
                f"lambda={lam:.2f}"
            )

        del mean_d, M2_d, sample_var, cross, coherent_d, update_d, update
        gc.collect()

    if return_history:
        return x, history

    return x



## 8. Single-image RL and summed-stack RL controls


In [ ]:

def single_image_rl(
    image,
    psf,
    iterations=20,
    background=0.0,
    init=None,
    eps=EPS,
):
    image = np.asarray(image, dtype=np.float32)
    psf = normalize_psf(psf)
    sensitivity = make_sensitivity(image.shape, psf)

    if init is None:
        x = image.copy()
        x -= np.float32(background)
        np.maximum(x, eps, out=x)
    else:
        x = np.asarray(init, dtype=np.float32).copy()
        np.maximum(x, eps, out=x)

    for _ in range(iterations):
        pred = conv2(x, psf)
        pred += np.float32(background)
        np.maximum(pred, eps, out=pred)

        ratio = image / pred
        correction = adjoint_conv2(ratio, psf)
        correction /= sensitivity

        x *= correction
        np.maximum(x, 0, out=x)

        del pred, ratio, correction

    return x


def streaming_sum(reader):
    summed = np.zeros(reader.shape, dtype=np.float32)

    for i in range(reader.n_frames):
        frame = reader.get_frame(i)
        summed += frame
        del frame

    return summed


## 9. User settings

In [ ]:

# ---------------------------
# USER SETTINGS
# ---------------------------

STACK_PATH = "synthetic_microtubule_stack.tif"

# PSF
USE_MEASURED_PSF = True
PSF_PATH = "synthetic_microtubule_psf.tif"

# Used only when USE_MEASURED_PSF = False
GAUSSIAN_PSF_SIZE = 21
GAUSSIAN_PSF_SIGMA = 1.8

ITERATIONS = 20

# 0.0 = standard multi-image RL
# 1.0 = full temporal-correlation update
# Recommended first tests: 0.25, 0.5, 0.75, 1.0
CORRELATION_STRENGTH = 0.75

# Background:
# "zero"       -> already background corrected
# "percentile" -> estimate one scalar background per frame
BACKGROUND_MODE = "percentile"
BACKGROUND_PERCENTILE = 5.0


## 10. Open the TIFF stack and PSF

In [ ]:

reader = TiffStackReader(STACK_PATH)

if USE_MEASURED_PSF:
    psf_user = tiff.imread(PSF_PATH).astype(np.float32)

    # If the PSF file contains singleton dimensions, remove them.
    psf_user = np.squeeze(psf_user)

    if psf_user.ndim != 2:
        raise ValueError(
            f"Measured PSF must be 2-D after squeezing; got {psf_user.shape}."
        )

    psf_user = normalize_psf(psf_user)

else:
    psf_user = gaussian_psf(
        size=GAUSSIAN_PSF_SIZE,
        sigma=GAUSSIAN_PSF_SIGMA,
    )

if BACKGROUND_MODE == "zero":
    backgrounds_user = np.zeros(reader.n_frames, dtype=np.float32)

elif BACKGROUND_MODE == "percentile":
    backgrounds_user = estimate_scalar_backgrounds(
        reader,
        percentile=BACKGROUND_PERCENTILE,
    )

else:
    raise ValueError("Unknown BACKGROUND_MODE.")

print("Backgrounds:")
print(backgrounds_user)
print("Mean background:", float(backgrounds_user.mean()))


## 11. Preview the data without loading the whole stack

In [ ]:

first_frame = reader.get_frame(0)
mean_image = streaming_mean(reader)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].imshow(first_frame, cmap="gray")
axes[0].set_title("First frame")
axes[0].axis("off")

axes[1].imshow(mean_image, cmap="gray")
axes[1].set_title("Temporal mean")
axes[1].axis("off")

axes[2].imshow(psf_user, cmap="gray")
axes[2].set_title("PSF")
axes[2].axis("off")

plt.tight_layout()
plt.show()

del first_frame



## 12. Run conventional and temporally correlated reconstructions

The same initial estimate is used for both methods to make the comparison cleaner.


In [ ]:

init_user = mean_image - np.float32(backgrounds_user.mean())
np.maximum(init_user, EPS, out=init_user)

# Standard multi-image RL
rec_multi = streaming_multi_image_rl(
    reader,
    psf_user,
    iterations=ITERATIONS,
    backgrounds=backgrounds_user,
    init=init_user,
    verbose=True,
)

# Temporally correlated multi-image RL
rec_temporal, history = temporally_correlated_multi_image_rl(
    reader,
    psf_user,
    iterations=ITERATIONS,
    backgrounds=backgrounds_user,
    correlation_strength=CORRELATION_STRENGTH,
    init=init_user,
    verbose=True,
    return_history=True,
)


## 13. Sum + RL control

In [ ]:

sum_image = streaming_sum(reader)

rec_sum = single_image_rl(
    sum_image,
    psf_user,
    iterations=ITERATIONS,
    background=float(backgrounds_user.sum()),
)

# Return summed reconstruction to per-frame intensity scale.
rec_sum /= np.float32(reader.n_frames)


## 14. Compare the results

In [ ]:

display_images = [
    (mean_image, "Raw temporal mean"),
    (rec_sum, "Sum + RL"),
    (rec_multi, "Standard multi-image RL"),
    (
        rec_temporal,
        f"Temporal-correlation RL (lambda={CORRELATION_STRENGTH})"
    ),
]

fig, axes = plt.subplots(1, 4, figsize=(17, 4))

# Use a common upper display percentile.
vmax = max(
    np.percentile(im, 99.7)
    for im, _ in display_images
)

for ax, (im, title) in zip(axes, display_images):
    ax.imshow(im, cmap="gray", vmin=0, vmax=vmax)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()


## 15. Temporal-correlation diagnostics

In [ ]:

plt.figure(figsize=(6, 4))
plt.plot(history["relative_change"])
plt.xlabel("Iteration")
plt.ylabel("Relative reconstruction change")
plt.title("Convergence")
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(history["mean_temporal_coherence"])
plt.xlabel("Iteration")
plt.ylabel("Mean temporal coherence")
plt.title("Temporal correction coherence")
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(history["positive_cross_fraction"])
plt.xlabel("Iteration")
plt.ylabel("Fraction of pixels with Q > 0")
plt.title("Positive temporal cross-product fraction")
plt.show()



## 16. Correlation-strength sweep

For large microscopy data, do **not keep all reconstructions in memory**.

This cell runs one value at a time, writes the result to disk immediately, and deletes it before moving to the next value.


In [ ]:

RUN_STRENGTH_SWEEP = False

if RUN_STRENGTH_SWEEP:

    strength_values = [0.0, 0.25, 0.5, 0.75, 1.0]

    output_folder = Path("temporal_RL_strength_sweep")
    output_folder.mkdir(exist_ok=True)

    for lam in strength_values:
        print("\n" + "=" * 60)
        print("Correlation strength:", lam)

        rec = temporally_correlated_multi_image_rl(
            reader,
            psf_user,
            iterations=ITERATIONS,
            backgrounds=backgrounds_user,
            correlation_strength=lam,
            init=init_user,
            verbose=False,
        )

        filename = output_folder / f"temporal_RL_lambda_{lam:.2f}.tif"
        tiff.imwrite(filename, rec.astype(np.float32))

        print("Saved:", filename)

        del rec
        gc.collect()


## 17. Save main results

In [ ]:

output_folder = Path("temporally_correlated_RL_output")
output_folder.mkdir(exist_ok=True)

tiff.imwrite(
    output_folder / "raw_temporal_mean.tif",
    mean_image.astype(np.float32)
)

tiff.imwrite(
    output_folder / "sum_plus_RL.tif",
    rec_sum.astype(np.float32)
)

tiff.imwrite(
    output_folder / "standard_multiimage_RL.tif",
    rec_multi.astype(np.float32)
)

tiff.imwrite(
    output_folder / "temporally_correlated_multiimage_RL.tif",
    rec_temporal.astype(np.float32)
)

print("Saved results to:", output_folder)



## 18. Close the TIFF reader

Run this when processing is finished.


In [ ]:

reader.close()
print("TIFF reader closed.")



# Interpretation and validation

The most important scientific controls are:

1. **single-frame RL**
2. **summed-stack + RL**
3. **standard multi-image RL**
4. **temporally correlated multi-image RL**

For identical repeated acquisitions with the same PSF and independent Poisson noise, standard multi-image RL and summed-stack RL should be very similar.

Therefore, to demonstrate that the temporal-correlation regularization is useful, method 4 should outperform methods 2 and 3 at the same total photon budget while preserving real structures and avoiding artifacts.

Recommended validation measurements include:

- SNR / CNR,
- background standard deviation,
- FRC or decorrelation resolution,
- line-profile FWHM,
- integrated fluorescence conservation,
- reconstruction error when a synthetic ground truth is available.

### Important assumptions

The temporal-correlation principle works best when:

- the specimen is static,
- frames are accurately registered,
- the PSF is stable,
- bleaching is small or corrected,
- noise between frames is approximately independent.

Motion, photobleaching, blinking, or biological dynamics can reduce temporal coherence of true structures and therefore suppress legitimate information.
